# 01-1. Train Separate-E2E

Shared-E2E와 GT-centered selection, patch adapter, auxiliary loss, aux weight, encoder/backbone gradient 경로는 동일하게 유지하고 BBox head parameter sharing만 제거하는 ablation입니다.

In [1]:
from pathlib import Path
import gc, importlib, sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (Path('D:/gt-super') / 'data').exists(): ROOT = Path('D:/gt-super')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import gt_aux.config as config_module
import gt_aux.data as data_module
import gt_aux.model as model_module
import gt_aux.eval as eval_module
import gt_aux.train as train_module
config_module = importlib.reload(config_module)
data_module = importlib.reload(data_module)
model_module = importlib.reload(model_module)
eval_module = importlib.reload(eval_module)
train_module = importlib.reload(train_module)

ExperimentConfig = config_module.ExperimentConfig
prepare_data, make_loaders = data_module.prepare_data, data_module.make_loaders
make_model = model_module.make_model
make_optimizer = train_module.make_optimizer
move_labels_to_device = train_module.move_labels_to_device
release_model = train_module.release_model
train_one_experiment = train_module.train_one_experiment

d:\gt-super\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 설정

Shared-E2E와 비교할 때 아래 설정을 동일하게 유지합니다. `SEEDS`의 값들이 순서대로 학습되며, `DATA_SEED`는 고정되어 모든 모델이 같은 데이터 split을 사용합니다.

In [2]:
EXPERIMENT = 'separate_e2e'
RUN_MODE = 'smoke'
SEEDS = [42, 43, 44]
DATA_SEED = 42
TRAIN_IMAGES, VAL_IMAGES, EPOCHS = 400, 100, 7
BATCH_SIZE, NUM_WORKERS = 2, 0
IMAGE_MIN_SIZE, IMAGE_MAX_SIZE = 384, 640
LEARNING_RATE, BACKBONE_LEARNING_RATE = 2e-4, 2e-5
WEIGHT_DECAY, GRAD_CLIP, AUX_WEIGHT = 1e-4, 0.1, 0.5
FEATURE_LEVEL, HORIZONTAL_FLIP_P = 0, 0.5
USE_AMP, DETERMINISTIC = None, True
SAVE_EPOCH_CHECKPOINTS = False
RESUME_FROM_BY_SEED = {}  # 예: {42: ROOT / 'cache/checkpoints/checkpoint_smoke_separate_e2e_seed42.pt'}

CONFIG = ExperimentConfig.for_run(
    ROOT, run_mode=RUN_MODE, seed=SEEDS[0], data_seed=DATA_SEED, experiments=[EXPERIMENT],
    train_images=TRAIN_IMAGES, val_images=VAL_IMAGES, epochs=EPOCHS,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    image_min_size=IMAGE_MIN_SIZE, image_max_size=IMAGE_MAX_SIZE,
    lr=LEARNING_RATE, backbone_lr=BACKBONE_LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP, base_aux_weight=AUX_WEIGHT, feature_level=FEATURE_LEVEL,
    horizontal_flip_p=HORIZONTAL_FLIP_P, use_amp=USE_AMP, deterministic=DETERMINISTIC,
    save_epoch_checkpoints=SAVE_EPOCH_CHECKPOINTS,
)
CONFIG.as_dict()

{'root': 'D:\\gt-super',
 'run_mode': 'smoke',
 'checkpoint': 'SenseTime/deformable-detr',
 'train_images': 400,
 'val_images': 100,
 'epochs': 7,
 'batch_size': 2,
 'num_workers': 0,
 'image_size': {'shortest_edge': 384, 'longest_edge': 640},
 'lr': 0.0002,
 'backbone_lr': 2e-05,
 'weight_decay': 0.0001,
 'grad_clip': 0.1,
 'base_aux_weight': 0.5,
 'feature_level': 0,
 'horizontal_flip_p': 0.5,
 'use_amp': True,
 'deterministic': True,
 'save_epoch_checkpoints': False,
 'device': 'cuda',
 'experiments': ['separate_e2e'],
 'seed': 42,
 'data_seed': 42}

In [3]:
BUNDLE = prepare_data(CONFIG)
print({'train_images': len(BUNDLE.train_records), 'val_images': len(BUNDLE.val_records), 'data_seed': DATA_SEED})

VOC XML: 100%|██████████| 3750/3750 [00:01<00:00, 3096.32it/s]

Full split: train=3000 (9180 objects), val=750 (2530 objects)
Current run: train=400, val=100
{'train_images': 400, 'val_images': 100, 'data_seed': 42}


## 구조·gradient flow 검증

한 batch로 auxiliary loss만 미분합니다. Aux Head·Adapter·Encoder·Backbone은 non-zero gradient, Main BBox Head·Decoder는 zero gradient여야 합니다. Separate-E2E에서는 공유 parameter가 없으므로 기존 shared-head gradient cosine은 정의하지 않습니다.

In [4]:
def parameter_group_grad_norm(loss, groups):
    names, parameters, sizes = [], [], []
    for name, group in groups.items():
        selected = [parameter for parameter in group if parameter.requires_grad]
        names.append(name); parameters.extend(selected); sizes.append(len(selected))
    gradients = torch.autograd.grad(loss, parameters, allow_unused=True)
    report, offset = {}, 0
    for name, size in zip(names, sizes):
        group_gradients = gradients[offset:offset + size]; offset += size
        squared = sum(float(gradient.detach().float().square().sum()) for gradient in group_gradients if gradient is not None)
        report[name] = squared ** 0.5
    return report

probe_model = probe_optimizer = probe_result = None
try:
    probe_model, _ = make_model(CONFIG, EXPERIMENT, SEEDS[0])
    main_head, aux_head = probe_model.shared_bbox_head, probe_model.aux_bbox_head
    main_parameters, aux_parameters = list(main_head.parameters()), list(aux_head.parameters())
    assert main_head is not aux_head
    assert all(main is not aux and main.data_ptr() != aux.data_ptr() for main, aux in zip(main_parameters, aux_parameters))
    assert all(torch.equal(main.detach(), aux.detach()) for main, aux in zip(main_parameters, aux_parameters))

    probe_optimizer = make_optimizer(probe_model, CONFIG)
    optimizer_parameter_ids = {id(parameter) for group in probe_optimizer.param_groups for parameter in group['params']}
    assert all(id(parameter) in optimizer_parameter_ids for parameter in aux_parameters)

    probe_loader, _ = make_loaders(CONFIG, BUNDLE, SEEDS[0])
    batch = next(iter(probe_loader))
    labels = move_labels_to_device(batch['labels'], CONFIG.device)
    probe_model.train()
    probe_result = probe_model(
        pixel_values=batch['pixel_values'].to(CONFIG.device),
        pixel_mask=batch['pixel_mask'].to(CONFIG.device), labels=labels, aux_weight=AUX_WEIGHT,
    )
    assert probe_result['aux_executed']
    flow_report = parameter_group_grad_norm(probe_result['aux_loss'], {
        'aux_bbox_head': probe_model.aux_bbox_head.parameters(),
        'patch_adapter': probe_model.adapter.parameters(),
        'encoder': probe_model.detector.model.encoder.parameters(),
        'backbone': probe_model.detector.model.backbone.parameters(),
        'main_bbox_head': probe_model.shared_bbox_head.parameters(),
        'decoder': probe_model.detector.model.decoder.parameters(),
    })
finally:
    del probe_result, probe_optimizer
    probe_model = release_model(probe_model)

for name in ['aux_bbox_head', 'patch_adapter', 'encoder', 'backbone']:
    assert flow_report[name] > 0, f'Expected aux gradient at {name}'
for name in ['main_bbox_head', 'decoder']:
    assert flow_report[name] == 0, f'Unexpected aux gradient at {name}'
print({'heads_are_independent': True, 'same_initial_values': True, 'aux_head_in_optimizer': True})
print(flow_report)

Loading weights: 100%|██████████| 545/545 [00:00<00:00, 18548.78it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

{'heads_are_independent': True, 'same_initial_values': True, 'aux_head_in_optimizer': True}
{'aux_bbox_head': 6.034960749154027, 'patch_adapter': 5.716884850860213, 'encoder': 20.752839102445712, 'backbone': 55.09149823966781, 'main_bbox_head': 0.0, 'decoder': 0.0}


## Separate-E2E 학습

In [5]:
seed_results = []
for seed in SEEDS:
    separate_model = None
    try:
        separate_model, separate_history, separate_gradients = train_one_experiment(
            CONFIG, BUNDLE, experiment=EXPERIMENT, seed=seed,
            resume_from=RESUME_FROM_BY_SEED.get(seed),
        )
        result = {
            'experiment': EXPERIMENT, 'seed': seed,
            'final_mAP': float(separate_history.iloc[-1]['map']),
            'checkpoint': str(CONFIG.checkpoint_path(EXPERIMENT, seed)),
        }
        seed_results.append(result)
        print(result)
    finally:
        separate_model = release_model(separate_model)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print({'completed_seeds': [result['seed'] for result in seed_results]})
seed_results


===== separate_e2e / seed=42 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 16560.26it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 50 batches


[phase] initial validation complete: mAP=0.0003, AP@0.5=0.0005
[phase] training epoch 1/7: 200 batches


separate_e2e e1:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\transformers\cuda\attention_back

...

{'epoch': 6, 'main_loss': 1.4193, 'aux_loss': 0.9649, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0621, 'map50': 0.107, 'map75': 0.0601}
[phase] training epoch 7/7: 200 batches


separate_e2e e7:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[phase] validating epoch 7/7: 50 batches


{'epoch': 7, 'main_loss': 1.3677, 'aux_loss': 0.879, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0743, 'map50': 0.1209, 'map75': 0.0821}
{'experiment': 'separate_e2e', 'seed': 44, 'final_mAP': 0.07434610277414322, 'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_separate_e2e_seed44.pt'}
{'completed_seeds': [42, 43, 44]}


[{'experiment': 'separate_e2e',
  'seed': 42,
  'final_mAP': 0.08701179921627045,
  'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_separate_e2e_seed42.pt'},
 {'experiment': 'separate_e2e',
  'seed': 43,
  'final_mAP': 0.07407944649457932,
  'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_separate_e2e_seed43.pt'},
 {'experiment': 'separate_e2e',
  'seed': 44,
  'final_mAP': 0.07434610277414322,
  'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_separate_e2e_seed44.pt'}]